In [ ]:
import os
import warnings
from datetime import datetime
from pathlib import Path
from typing import List, Dict, Tuple, Optional
import glob

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import RobustScaler, QuantileTransformer
from sklearn.decomposition import PCA
import umap

warnings.filterwarnings("ignore")


In [ ]:
# ================================================================================
# CONFIGURATION
# ================================================================================
class Config:
    # Paths
    DATA_FOLDER = r"D:\2025_12_19 CRISPRi Reference Plate Imaging\Cell_Features"
    PARQUET_PATTERN = "micromorph_cell_measurements_P*.parquet"

    # Features for analysis (reused from original script)
    FEATURES = [
        "area_um2",
        "perimeter_um",
        "length_um",
        "width_um",
        "aspect_ratio",
        "roundness",
        "solidity",
        "eccentricity"
    ]

    # Analysis parameters
    RANDOM_STATE = 42
    FIGURE_SIZE_WIDE = (20, 12)
    FIGURE_SIZE_STANDARD = (13.333, 7.5)
    DPI = 300

    # UMAP parameters (can be adjusted based on grid search results)
    UMAP_N_NEIGHBORS = 15
    UMAP_MIN_DIST = 0.1

    # Expected number of FOVs per plate
    EXPECTED_FOVS_PER_PLATE = 2016

    # Viridis colors for 6 plates (sampled from viridis colormap)
    PLATE_COLORS = None  # Will be generated dynamically

In [ ]:
# ================================================================================
# UTILITY FUNCTIONS
# ================================================================================
def setup_analysis_folder(base_folder: str) -> str:
    """Create timestamped analysis output folder."""
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    analysis_folder = os.path.join(base_folder, f"MultiPlate_Analysis_{timestamp}")
    os.makedirs(analysis_folder, exist_ok=True)
    return analysis_folder


def generate_plate_colors(n_plates: int) -> Dict[str, str]:
    """Generate hex colors from viridis colormap for each plate."""
    from matplotlib.colors import rgb2hex
    cmap = plt.cm.viridis
    colors = {}
    for i in range(n_plates):
        # Sample evenly across viridis
        rgb = cmap(i / max(n_plates - 1, 1))
        colors[f"P{i+1}"] = rgb2hex(rgb)
    return colors


def parse_gene_subgroup(label: str) -> Tuple[str, Optional[str]]:
    """Parse gene_subgroup format (e.g., 'mrcA_1' -> ('mrcA', '1'))."""
    if pd.isna(label):
        return np.nan, None
    if '_' in str(label) and str(label) != 'WT':
        parts = str(label).rsplit('_', 1)
        if len(parts) == 2 and parts[1] in ['1', '2', '3']:
            return parts[0], parts[1]
    return str(label), None

In [ ]:
# ================================================================================
# DATA PROCESSING
# ================================================================================
class MultiPlateDataProcessor:
    """Handles loading and preprocessing of multiple plates."""

    def __init__(self, data_folder: str, parquet_pattern: str):
        self.data_folder = data_folder
        self.parquet_pattern = parquet_pattern
        self.plate_maps = {}

    def find_plate_map_excel(self, plate_id: str) -> Optional[str]:
        """Find Excel plate map file for a specific plate."""
        # Search for any .xlsx or .xls file containing the plate_id
        # Pattern: *_P1.xlsx, *_P2.xlsx, etc.

        patterns = [
            f"*_{plate_id}.xlsx",
            f"*_{plate_id}.xls"
        ]

        for pattern in patterns:
            files = glob.glob(os.path.join(self.data_folder, pattern))
            if files:
                return files[0]  # Return first match

        return None

    def load_plate_map(self, plate_id: str) -> Optional[pd.DataFrame]:
        """Load plate map Excel file for a specific plate."""
        excel_file = self.find_plate_map_excel(plate_id)

        if excel_file:
            try:
                plate_map = pd.read_excel(excel_file, header=None)
                print(f"    âœ“ Loaded plate map: {os.path.basename(excel_file)} ({plate_map.shape[0]}Ã—{plate_map.shape[1]})")
                return plate_map
            except Exception as e:
                print(f"    âš  Error loading {os.path.basename(excel_file)}: {e}")
                return None
        else:
            print(f"    âš  No Excel plate map found for {plate_id}")
            # List available Excel files for debugging
            excel_files = glob.glob(os.path.join(self.data_folder, "*.xlsx"))
            if excel_files:
                print(f"    Available Excel files in folder:")
                for f in excel_files[:5]:  # Show first 5
                    print(f"      - {os.path.basename(f)}")
            return None

    def assign_well_labels(self, data: pd.DataFrame, plate_map: pd.DataFrame) -> pd.DataFrame:
        """Map well positions to gene labels using plate map (reused from original)."""
        def get_label(well: str) -> str:
            if pd.isna(well) or len(str(well)) < 2:
                return np.nan
            well_str = str(well)
            row = ord(well_str[0].upper()) - ord('A')
            try:
                col = int(well_str[1:]) - 1
            except ValueError:
                return np.nan

            if 0 <= row < plate_map.shape[0] and 0 <= col < plate_map.shape[1]:
                return plate_map.iat[row, col]
            return np.nan

        data["Label"] = data["Well"].map(get_label).astype(str)
        return data

    def load_all_plates(self) -> pd.DataFrame:
        """Load all parquet files and add plate identifier."""
        parquet_files = sorted(glob.glob(os.path.join(self.data_folder, self.parquet_pattern)))

        if len(parquet_files) == 0:
            raise FileNotFoundError(f"No files matching {self.parquet_pattern} in {self.data_folder}")

        print(f"\n{'='*80}")
        print(f"LOADING {len(parquet_files)} PLATES")
        print(f"{'='*80}")

        all_data = []

        for file_path in parquet_files:
            # Extract plate ID from filename (e.g., P1, P2, ...)
            filename = os.path.basename(file_path)
            plate_id = filename.split('_')[-1].replace('.parquet', '')

            print(f"\n{plate_id}: {filename}")
            print(f"  Loading parquet...", end=' ')
            data = pd.read_parquet(file_path, engine="pyarrow")
            print(f"âœ“ {len(data):,} cells")

            # Add plate identifier
            data['Plate'] = plate_id

            # Load corresponding plate map
            print(f"  Looking for plate map...", end=' ')
            plate_map = self.load_plate_map(plate_id)

            if plate_map is not None:
                self.plate_maps[plate_id] = plate_map
                # Assign labels from plate map
                print(f"  Assigning labels from plate map...", end=' ')
                data = self.assign_well_labels(data, plate_map)
                n_labeled = data['Label'].notna().sum()
                n_valid = (data['Label'] != 'nan').sum()
                print(f"âœ“ {n_valid:,} cells with valid labels")
            else:
                print(f"  âŒ Skipping {plate_id} - no plate map found")
                continue

            all_data.append(data)

        if len(all_data) == 0:
            raise ValueError("No valid plates loaded! Check plate map files.")

        combined_data = pd.concat(all_data, ignore_index=True)
        print(f"\n{'='*80}")
        print(f"âœ“ Total: {len(combined_data):,} cells from {len(all_data)} plates")
        print(f"{'='*80}")

        return combined_data

    def preprocess_features(self, data: pd.DataFrame) -> pd.DataFrame:
        """Clip outliers and ensure numeric types (reused from original)."""
        print(f"\n{'='*80}")
        print(f"PREPROCESSING FEATURES")
        print(f"{'='*80}")

        feature_ranges = {
            "roundness": (0, 1),
            "solidity": (0, 1),
            "eccentricity": (0, 1),
            "aspect_ratio": (1, 20)
        }

        for feat, (low, high) in feature_ranges.items():
            if feat in data.columns:
                data[feat] = data[feat].clip(low, high)

        for feat in Config.FEATURES:
            if feat in data.columns:
                data[feat] = pd.to_numeric(data[feat], errors='coerce')

        # Remove rows with non-finite values in any feature
        initial_count = len(data)
        for feat in Config.FEATURES:
            if feat in data.columns:
                data = data[np.isfinite(data[feat])]

        print(f"âœ“ Removed {initial_count - len(data):,} cells with invalid features")

        return data

    def assign_gene_subgroup(self, data: pd.DataFrame) -> pd.DataFrame:
        """Parse gene and subgroup from Label column."""
        print(f"\n{'='*80}")
        print(f"PARSING GENE AND SUBGROUP LABELS")
        print(f"{'='*80}")

        data[['Gene', 'Subgroup']] = data['Label'].apply(
            lambda x: pd.Series(parse_gene_subgroup(x))
        )

        # Remove rows with NaN or 'nan' string genes
        initial_count = len(data)
        data = data[data['Gene'].notna()].copy()
        data = data[data['Gene'] != 'nan'].copy()

        print(f"âœ“ Removed {initial_count - len(data):,} cells with invalid gene labels")

        genes = sorted(data['Gene'].unique())
        plates = sorted(data['Plate'].unique())

        print(f"\nGenes detected: {genes}")
        print(f"Plates detected: {plates}")

        # Print distribution
        print(f"\nGene distribution per plate:")
        for plate in plates:
            plate_data = data[data['Plate'] == plate]
            gene_counts = plate_data['Gene'].value_counts()
            print(f"  {plate}: {len(plate_data):,} cells, {len(gene_counts)} genes")

        return data

In [ ]:
# ================================================================================
# UMAP ANALYSIS
# ================================================================================
class MultiPlateUMAPAnalyzer:
    """Handles UMAP dimensionality reduction for multi-plate data."""

    def __init__(self, analysis_folder: str, plate_colors: Dict[str, str]):
        self.analysis_folder = analysis_folder
        self.plate_colors = plate_colors

    def _prepare_fov_data(self, data: pd.DataFrame, features: List[str]) -> pd.DataFrame:
        """Aggregate cells to FOV level using median (reused from original)."""
        print(f"\n{'='*80}")
        print(f"AGGREGATING TO FOV LEVEL")
        print(f"{'='*80}")

        available_features = [f for f in features if f in data.columns]

        # Calculate median features per FOV
        grouping_cols = ["Plate", "Well", "Point", "filename"]
        fov_medians = data.groupby(grouping_cols)[available_features].median().reset_index()

        # Count cells per FOV/gene/subgroup combination
        fov_gene_counts = data.groupby(
            grouping_cols + ["Label", "Gene", "Subgroup"],
            dropna=False  # Keep WT which has Subgroup=None
        ).size().reset_index(name='cell_count')

        # For each FOV, find the dominant gene (most cells)
        idx = fov_gene_counts.groupby(grouping_cols)['cell_count'].idxmax()
        fov_dominant = fov_gene_counts.loc[idx, grouping_cols + ["Label", "Gene", "Subgroup"]]

        # Merge median features with dominant gene labels
        fov_data = fov_medians.merge(fov_dominant, on=grouping_cols)

        print(f"âœ“ {len(fov_data):,} FOVs aggregated")

        # Print distribution
        plate_counts = fov_data['Plate'].value_counts().sort_index()
        for plate, count in plate_counts.items():
            print(f"  {plate}: {count:,} FOVs")

        return fov_data

    def _preprocess_for_umap(self, X: pd.DataFrame) -> np.ndarray:
        """Apply robust scaling and quantile transformation (reused from original)."""
        scaler = RobustScaler()
        X_robust = scaler.fit_transform(X)

        quantile = QuantileTransformer(
            n_quantiles=1000,
            output_distribution='normal',
            random_state=Config.RANDOM_STATE
        )
        X_scaled = quantile.fit_transform(X_robust)

        n_components = min(10, X_scaled.shape[1])
        pca = PCA(n_components=n_components, random_state=Config.RANDOM_STATE)
        X_pca = pca.fit_transform(X_scaled)

        return X_pca

    def compute_umap_embedding(self, fov_data: pd.DataFrame, features: List[str]) -> Tuple[np.ndarray, pd.DataFrame]:
        """Compute UMAP embedding for all FOVs."""
        print(f"\n{'='*80}")
        print(f"COMPUTING UMAP EMBEDDING")
        print(f"{'='*80}")
        print(f"Parameters: n_neighbors={Config.UMAP_N_NEIGHBORS}, min_dist={Config.UMAP_MIN_DIST}")

        available_features = [f for f in features if f in fov_data.columns]
        X = fov_data[available_features].dropna().astype(np.float32)
        X_pca = self._preprocess_for_umap(X)

        reducer = umap.UMAP(
            n_neighbors=Config.UMAP_N_NEIGHBORS,
            min_dist=Config.UMAP_MIN_DIST,
            spread=0.5,
            metric='euclidean',
            random_state=Config.RANDOM_STATE,
            n_jobs=-1
        )

        embedding = reducer.fit_transform(X_pca)

        print(f"âœ“ UMAP embedding computed: {embedding.shape}")

        return embedding, fov_data

    def plot_umap_by_gene_colored_by_plate(self, embedding: np.ndarray, fov_data: pd.DataFrame):
        """UMAP aggregated by Gene, colored by Plate."""
        print(f"\n{'='*80}")
        print(f"GENERATING UMAP: Gene Aggregation (Colored by Plate)")
        print(f"{'='*80}")

        output_folder = os.path.join(self.analysis_folder, "UMAP_Gene_ByPlate")
        os.makedirs(output_folder, exist_ok=True)

        fig, ax = plt.subplots(1, 1, figsize=Config.FIGURE_SIZE_WIDE)

        # Plot each plate with its color
        plates = sorted(fov_data['Plate'].unique())
        for plate in plates:
            plate_mask = fov_data['Plate'] == plate
            plate_data = fov_data[plate_mask]

            indices = plate_data.index
            emb_indices = [i for i, idx in enumerate(fov_data.index) if idx in indices]

            if len(emb_indices) > 0:
                ax.scatter(
                    embedding[emb_indices, 0],
                    embedding[emb_indices, 1],
                    c=self.plate_colors[plate],
                    s=40,
                    alpha=0.7,
                    marker='o',
                    edgecolors='white',
                    linewidth=0.5,
                    label=f'{plate} (n={len(emb_indices)})',
                    rasterized=True
                )

        ax.set_title(
            f"UMAP: Gene Aggregation (Colored by Plate)\n{len(fov_data):,} FOVs from {len(plates)} plates",
            fontsize=16,
            fontweight='bold',
            pad=20
        )
        ax.set_xlabel("UMAP 1", fontsize=12)
        ax.set_ylabel("UMAP 2", fontsize=12)
        ax.grid(True, alpha=0.3)
        ax.legend(loc='best', fontsize=10, framealpha=0.9)

        plt.tight_layout()
        plt.savefig(
            os.path.join(output_folder, "UMAP_Gene_ColoredByPlate.png"),
            dpi=Config.DPI,
            bbox_inches="tight",
            facecolor='white'
        )
        plt.close()

        print(f"âœ“ Saved: UMAP_Gene_ColoredByPlate.png")

    def plot_umap_by_subgroup_colored_by_plate(self, embedding: np.ndarray, fov_data: pd.DataFrame):
        """UMAP aggregated by Subgroup, colored by Plate."""
        print(f"\n{'='*80}")
        print(f"GENERATING UMAP: Subgroup Aggregation (Colored by Plate)")
        print(f"{'='*80}")

        output_folder = os.path.join(self.analysis_folder, "UMAP_Subgroup_ByPlate")
        os.makedirs(output_folder, exist_ok=True)

        fig, ax = plt.subplots(1, 1, figsize=Config.FIGURE_SIZE_WIDE)

        # Define markers for subgroups
        subgroup_markers = {'1': 'o', '2': 's', '3': '^', None: 'o'}

        # Plot each plate with its color, varying marker by subgroup
        plates = sorted(fov_data['Plate'].unique())
        for plate in plates:
            plate_mask = fov_data['Plate'] == plate
            plate_data = fov_data[plate_mask]

            # Get unique subgroups in this plate
            subgroups = plate_data['Subgroup'].unique()

            for subgroup in subgroups:
                if pd.isna(subgroup):
                    subgroup_mask = plate_data['Subgroup'].isna()
                    marker = subgroup_markers[None]
                    label_suffix = ""
                else:
                    subgroup_mask = plate_data['Subgroup'] == subgroup
                    marker = subgroup_markers.get(subgroup, 'o')
                    label_suffix = f"_{subgroup}"

                indices = plate_data[subgroup_mask].index
                emb_indices = [i for i, idx in enumerate(fov_data.index) if idx in indices]

                if len(emb_indices) > 0:
                    ax.scatter(
                        embedding[emb_indices, 0],
                        embedding[emb_indices, 1],
                        c=self.plate_colors[plate],
                        s=40,
                        alpha=0.7,
                        marker=marker,
                        edgecolors='white',
                        linewidth=0.5,
                        label=f'{plate}{label_suffix} (n={len(emb_indices)})',
                        rasterized=True
                    )

        ax.set_title(
            f"UMAP: Subgroup Aggregation (Colored by Plate)\n{len(fov_data):,} FOVs from {len(plates)} plates",
            fontsize=16,
            fontweight='bold',
            pad=20
        )
        ax.set_xlabel("UMAP 1", fontsize=12)
        ax.set_ylabel("UMAP 2", fontsize=12)
        ax.grid(True, alpha=0.3)
        ax.legend(loc='best', fontsize=8, framealpha=0.9, ncol=2)

        plt.tight_layout()
        plt.savefig(
            os.path.join(output_folder, "UMAP_Subgroup_ColoredByPlate.png"),
            dpi=Config.DPI,
            bbox_inches="tight",
            facecolor='white'
        )
        plt.close()

        print(f"âœ“ Saved: UMAP_Subgroup_ColoredByPlate.png")

    def plot_plate_focused_umaps(self, embedding: np.ndarray, fov_data: pd.DataFrame):
        """Generate one UMAP per plate with that plate highlighted."""
        print(f"\n{'='*80}")
        print(f"GENERATING PLATE-FOCUSED UMAPs")
        print(f"{'='*80}")

        output_folder = os.path.join(self.analysis_folder, "UMAP_PlateFocused")
        os.makedirs(output_folder, exist_ok=True)

        plates = sorted(fov_data['Plate'].unique())

        # Calculate consistent axis limits across all plots
        x_min, x_max = embedding[:, 0].min(), embedding[:, 0].max()
        y_min, y_max = embedding[:, 1].min(), embedding[:, 1].max()
        x_margin = (x_max - x_min) * 0.05
        y_margin = (y_max - y_min) * 0.05

        for plate_idx, highlight_plate in enumerate(plates, 1):
            print(f"  [{plate_idx}/{len(plates)}] Highlighting {highlight_plate}...", end=' ')

            fig, ax = plt.subplots(1, 1, figsize=Config.FIGURE_SIZE_WIDE)

            # Plot all other plates in grey (background)
            for plate in plates:
                if plate != highlight_plate:
                    plate_mask = fov_data['Plate'] == plate
                    plate_data = fov_data[plate_mask]

                    indices = plate_data.index
                    emb_indices = [i for i, idx in enumerate(fov_data.index) if idx in indices]

                    if len(emb_indices) > 0:
                        ax.scatter(
                            embedding[emb_indices, 0],
                            embedding[emb_indices, 1],
                            c='lightgrey',
                            s=20,
                            alpha=0.3,
                            marker='o',
                            edgecolors='none',
                            rasterized=True
                        )

            # Plot highlighted plate
            highlight_mask = fov_data['Plate'] == highlight_plate
            highlight_data = fov_data[highlight_mask]

            # Separate WT and non-WT
            wt_mask = highlight_data['Gene'] == 'WT'

            # Plot non-WT with plate color
            non_wt_data = highlight_data[~wt_mask]
            if len(non_wt_data) > 0:
                indices = non_wt_data.index
                emb_indices = [i for i, idx in enumerate(fov_data.index) if idx in indices]

                if len(emb_indices) > 0:
                    ax.scatter(
                        embedding[emb_indices, 0],
                        embedding[emb_indices, 1],
                        c=self.plate_colors[highlight_plate],
                        s=50,
                        alpha=0.8,
                        marker='o',
                        edgecolors='white',
                        linewidth=0.8,
                        label=f'{highlight_plate} (n={len(emb_indices)})',
                        zorder=10,
                        rasterized=True
                    )

            # Plot WT in black
            wt_data = highlight_data[wt_mask]
            if len(wt_data) > 0:
                indices = wt_data.index
                emb_indices = [i for i, idx in enumerate(fov_data.index) if idx in indices]

                if len(emb_indices) > 0:
                    ax.scatter(
                        embedding[emb_indices, 0],
                        embedding[emb_indices, 1],
                        c='black',
                        s=50,
                        alpha=0.8,
                        marker='o',
                        edgecolors='white',
                        linewidth=0.8,
                        label=f'{highlight_plate} WT (n={len(emb_indices)})',
                        zorder=11,
                        rasterized=True
                    )

            # Set consistent axes
            ax.set_xlim(x_min - x_margin, x_max + x_margin)
            ax.set_ylim(y_min - y_margin, y_max + y_margin)

            ax.set_title(
                f"UMAP: {highlight_plate} Highlighted\n{len(fov_data):,} FOVs from {len(plates)} plates (others in grey)",
                fontsize=16,
                fontweight='bold',
                pad=20
            )
            ax.set_xlabel("UMAP 1", fontsize=12)
            ax.set_ylabel("UMAP 2", fontsize=12)
            ax.grid(True, alpha=0.3)
            ax.legend(loc='best', fontsize=10, framealpha=0.9)

            plt.tight_layout()
            plt.savefig(
                os.path.join(output_folder, f"UMAP_Highlighted_{highlight_plate}.png"),
                dpi=Config.DPI,
                bbox_inches="tight",
                facecolor='white'
            )
            plt.close()

            print(f"âœ“")

        print(f"âœ“ All plate-focused UMAPs complete")

In [ ]:
# ================================================================================
# MAIN PIPELINE
# ================================================================================
def main():
    """Execute multi-plate UMAP analysis pipeline."""
    print(f"\n{'='*80}")
    print(f"MULTI-PLATE CRISPRi UMAP ANALYSIS PIPELINE")
    print(f"{'='*80}")

    # Setup analysis folder
    analysis_folder = setup_analysis_folder(Config.DATA_FOLDER)
    print(f"\nâœ“ Analysis folder: {analysis_folder}")

    # Initialize processor
    processor = MultiPlateDataProcessor(
        Config.DATA_FOLDER,
        Config.PARQUET_PATTERN
    )

    # Load all plates (with plate map assignment)
    data = processor.load_all_plates()

    # Parse gene and subgroup
    data = processor.assign_gene_subgroup(data)

    # Preprocess features
    data = processor.preprocess_features(data)

    # Generate plate colors
    plates = sorted(data['Plate'].unique())
    Config.PLATE_COLORS = generate_plate_colors(len(plates))
    print(f"\n{'='*80}")
    print(f"PLATE COLOR MAPPING (viridis)")
    print(f"{'='*80}")
    for plate, color in Config.PLATE_COLORS.items():
        print(f"  {plate}: {color}")

    # Initialize UMAP analyzer
    umap_analyzer = MultiPlateUMAPAnalyzer(analysis_folder, Config.PLATE_COLORS)

    # Prepare FOV-level data
    fov_data = umap_analyzer._prepare_fov_data(data, Config.FEATURES)

    # Compute UMAP embedding (single embedding for all analyses)
    embedding, fov_data = umap_analyzer.compute_umap_embedding(fov_data, Config.FEATURES)

    # Save embedding data
    fov_data_with_umap = fov_data.copy()
    fov_data_with_umap['UMAP_1'] = embedding[:, 0]
    fov_data_with_umap['UMAP_2'] = embedding[:, 1]
    fov_data_with_umap.to_csv(
        os.path.join(analysis_folder, "fov_umap_multiplate.csv"),
        index=False
    )
    print(f"\nâœ“ Saved FOV data with UMAP coordinates: fov_umap_multiplate.csv")

    # Generate all plots
    print(f"\n{'='*80}")
    print(f"GENERATING VISUALIZATIONS")
    print(f"{'='*80}")

    # 1. Gene aggregation, colored by plate
    umap_analyzer.plot_umap_by_gene_colored_by_plate(embedding, fov_data)

    # 2. Subgroup aggregation, colored by plate
    umap_analyzer.plot_umap_by_subgroup_colored_by_plate(embedding, fov_data)

    # 3. Plate-focused UMAPs
    umap_analyzer.plot_plate_focused_umaps(embedding, fov_data)

    print(f"\n{'='*80}")
    print(f"âœ“ ANALYSIS COMPLETE")
    print(f"{'='*80}")
    print(f"Results saved to: {analysis_folder}")
    print(f"\nGenerated outputs:")
    print(f"  â€¢ UMAP_Gene_ByPlate/UMAP_Gene_ColoredByPlate.png")
    print(f"  â€¢ UMAP_Subgroup_ByPlate/UMAP_Subgroup_ColoredByPlate.png")
    print(f"  â€¢ UMAP_PlateFocused/UMAP_Highlighted_P*.png ({len(plates)} files)")
    print(f"  â€¢ fov_umap_multiplate.csv (FOV data with UMAP coordinates)")


if __name__ == "__main__":
    main()